# 06 - Results export (public-safe artifacts only)

**Private compute session - not a dataset mirror.** Raw BraTS data come only from the official TCIA source into ephemeral session storage (`WORK`) and are never written to notebook output, a Kaggle Dataset, GitHub or the website. B1 basis: owner-approved alternative (amendment v1.0-A1); no TCIA authorization is claimed. Enable **GPU** and **Internet** in the notebook settings. Never paste credentials into a cell; use Kaggle Secrets if a private repository needs a token.

In [ ]:
# PARAMETERS - set before running (no secrets here)
REPO_URL = ""   # https URL of this research repository (GitHub)
COMMIT = ""     # exact commit SHA or tag to run (never a moving branch)
WORK = "/tmp/brats"                     # ephemeral session storage (raw data stay here)
EXPORT = "/kaggle/working/export"       # records/reports/artifacts only - never raw data
assert REPO_URL, "set REPO_URL"
assert COMMIT, "set COMMIT"


In [ ]:
import os
import subprocess

os.makedirs(WORK, exist_ok=True)
os.makedirs(EXPORT, exist_ok=True)
!git clone --quiet {REPO_URL} /kaggle/working/repo
%cd /kaggle/working/repo
!git checkout --quiet {COMMIT}
!git rev-parse HEAD
!pip install --quiet -e ".[io,nnunet]"
!brats-uncertainty verify-protocol

Validates every `*.metrics.json` (synthetic records are refused) and copies only `.json .csv .md .png .svg .txt` into `export/public`. Checkpoints, images, labels, archives and raw data are skipped and listed - never copied. The owner downloads `export/public`, runs `brats-uncertainty export-artifacts` again into the repository's `results/`, reviews and commits.

In [ ]:
# STOP if there are no actual result artifacts: nothing is ever invented
RESULTS = "/kaggle/working/results"
has_results = os.path.isdir(RESULTS) and any(
    name.endswith(".metrics.json") for _, _, names in os.walk(RESULTS) for name in names
)
assert has_results, "no *.metrics.json result artifacts exist: stop here"

In [ ]:
!brats-uncertainty export-artifacts --source {RESULTS} --dest {EXPORT}/public